In [1]:
import pandas as pd
import numpy as np

In [2]:
events = pd.read_parquet(f"data/events.parquet")
races = pd.read_parquet(f"data/races.parquet")
runners = pd.read_parquet(f"data/runners.parquet")

In [3]:
fbase = "output/data"

# Pre-prep data

## Location Names

## Normalised Probabilities

In [4]:
book_pct = runners.groupby("race_id").implied_odds.sum()
runners["normalised_odds"] = runners.implied_odds/runners.race_id.map(book_pct)

## Test Statistics

In [5]:
runners["returns"] = runners.bsp * runners.win_lose
runners["brier"]   = np.square(runners.normalised_odds - runners.win_lose)
runners["log_loss"] = -(
    np.log(runners.normalised_odds)*runners.win_lose +
    np.log(1-runners.normalised_odds)*(1-runners.win_lose)
)

/home/mabel/.pyenv/versions/3.14.2/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Bsp Binning Range

In [6]:
ticks = np.concat((
    np.arange(1, 2, 0.01),
    np.arange(2,3,0.02),
    np.arange(3,4,0.05),
    np.arange(4,6,0.1),
    np.arange(6,10,0.2),
    np.arange(10,20,0.5),
    np.arange(20,30,1),
    np.arange(30,50,2),
    np.arange(50,100,5),
    np.arange(100,1000,10),
    [1001]
))
tick_range=[0,2,3,4,6,10,20,30,50,100,1001]
tick_size=[0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10]
#ticks=1000
t = {
    1: 0.01,
    2: 0.02,
    3: 0.05,
    4: 0.1,
    6: 0.2,
    10: 0.5,
    20: 1,
    30: 2,
    50: 5,
    100: 10
}

# Favourite-Longshot Bias

## All

In [7]:
g = runners.groupby(pd.cut(runners.bsp, bins=ticks), observed=False)

In [8]:
all_df = pd.DataFrame({"mean": g.win_lose.mean(), "count": g.size(), "volume": g.pptradedvol.sum()})

In [9]:
all_df["tick_size"] = np.array(tick_size)[pd.cut(all_df.index.categories.right, bins=tick_range, labels=False)]
#df = df.rolling(5).mean()
all_df = all_df.rolling(5, min_periods=1).mean()
all_df["l"] = all_df.index.categories.left
all_df["r"] = all_df.index.categories.right
all_df.index=all_df.l

In [10]:
all_df.to_csv(f"{fbase}/all-fl.csv")

In [11]:
all_df["count"]

l
1.00         2.0
1.01         1.0
1.02         1.0
1.03         2.0
1.04         3.6
           ...  
950.00     244.8
960.00     239.8
970.00     244.0
980.00     254.8
990.00    3475.4
Name: count, Length: 350, dtype: float64

## Groups

per race stats

In [12]:
rg = runners.groupby("race_id")

nraces = races.assign(
    ppvol=rg.pptradedvol.sum(),
    nrunners=rg.size(),
    overround=rg.implied_odds.sum()
)

add groups that are needed

In [13]:
df = runners.join(nraces, on="race_id").join(events, on="event_id")
df["hour"] = df.race_time.dt.hour
df["race_ppvol"] = df.race_id.map(pd.qcut(nraces.ppvol, q=10))
df["overround"] = df.race_id.map(pd.qcut(nraces.overround, q=10))

In [14]:
cats = {
    "race_no": ("race_no", [1,2,3,4,5,6,7,8]),
    "distance_f": ("distance_f", df.distance_f.value_counts().head(10).index),
    "hour": ("hour", np.arange(12,20,1)),
    "race_type": ("race_type", df.race_type.value_counts().head(10).index),
    "location": ("location", df.location.value_counts().head(10).index),
    "nrunners": ("nrunners", df.nrunners.value_counts().head(10).index),
    "race_ppvol": ("race_ppvol", slice(None)),
    "overround": ("overround", slice(None))
}

In [15]:
for c in cats:
    for a, v in [("mean", "win_lose"), ("count", "win_lose"), ("sum", "pptradedvol")]:
        pt = df.pivot_table(
            index=pd.cut(df.bsp, bins=ticks),
            columns=cats[c][0],
            values=v,
            aggfunc=a,
            observed=False,
            dropna=False
        ).loc[:, cats[c][1]]
        pt = pt.reindex(sorted(pt.columns), axis=1)
        pt.columns = pt.columns.map(str).str.replace(",", ";")
        ts = np.array(tick_size)[pd.cut(pt.index.categories.right, bins=tick_range, labels=False)]
        pt.insert(0, 'tick_size', ts)
        #pt=pt.groupby("tick_size", as_index=False).rolling(5, min_periods=1).mean()
        pt = pt.rolling(5).mean()
        l = pt.index.categories.left
        r = pt.index.categories.right
        
        pt.index=l
        pt.to_csv(f"{fbase}/{c}-fl-{a}.csv")

# Test Statistics

## All

In [16]:
all_brier = runners.groupby("race_id").brier.mean().mean()
all_ll = runners.groupby("race_id").log_loss.mean().mean()

In [17]:
print(f"Brier: {all_brier}")
print(f"Log-Loss: {all_ll}")

Brier: 0.09461983594838873
Log-Loss: 0.31573284904459953


In [18]:
b = []
l = []
r = np.arange(0.02,1,0.02)
for x in r:
    b.append(np.square(x - runners.win_lose).groupby(runners.race_id).mean().mean())
    l.append(-(np.log(x)*runners.win_lose + np.log(1-x)*(1-runners.win_lose)).groupby(runners.race_id).mean().mean())
#d = pd.DataFrame(d)
ndf = pd.DataFrame({"brier": 1-b/all_brier, "log_loss": 1-l/all_ll}, index=r)
ndf.to_csv(f"{fbase}/all_ss_random.csv")

## Groups

In [19]:
for c in cats:
    b = df.groupby(["race_id", cats[c][0]]).brier.mean().groupby(level=1).mean()
    l = df.groupby(["race_id", cats[c][0]]).log_loss.mean().groupby(level=1).mean()

    f = pd.DataFrame({"brier": 1-b/all_brier,
                      "log_loss": 1-l/all_ll,
                      "count": df.groupby(["race_id", cats[c][0]]).size().groupby(level=1).size(),
                      "volume": df.groupby(cats[c][0]).pptradedvol.sum()})
    f = f.loc[cats[c][1], :]
    f = f.sort_index()
    f.to_csv(f"{fbase}/{c}-acc.csv")

# Trading

In [20]:
all_favourites = df.sort_values("bsp").groupby("race_id").head(1)

In [21]:
def create_bets(slip, mode="abs"):
    bets=slip
    # generate timeline
    #sort = ["race_time"] if not group else [group, "race_time"]
    bets = bets.join(races[["race_time"]], on="race_id", how="inner").sort_values("race_time")
    # pnl
    if mode == "abs":
        bets["profit"] = (bets.bsp * bets.win_lose -1) * bets.liability
        #bets["cum_profit"] = bets.groupby(group).profit.cumsum()
    elif mode == "pct":
        bets["profit"] = bets.liability*(bets.bsp*bets.win_lose-1)+1
    
    return bets

In [22]:
def bet_on_favourite(group=None):
    favs = all_favourites
    cols = ["race_id", "win_lose", "bsp"]
    if group:
        cols += [group]
    slip = favs[cols]
    slip = slip.assign(liability=1)

    return create_bets(slip)

In [23]:
def hedge_on_favourite(group=None):
    favs = all_favourites
    cols = ["race_id", "win_lose", "bsp"]
    if group:
        cols += [group]
    slip = favs[cols]
    slip = slip.assign(liability=favs.implied_odds)

    return create_bets(slip)

In [24]:
def bet_kelly(group=None, window_size=1000, bet_size=1):
    favs = all_favourites.sort_values("race_time")
    if group:
        p = favs.groupby(group).win_lose.transform(lambda s: s.rolling(window_size).mean())
    else:
        p = favs.win_lose.rolling(window_size).mean()
    q = 1-p
    b = favs.bsp-1

    cols = ["race_id", "win_lose", "bsp"]
    if group:
        cols += [group]
    slip = favs[cols]
    slip = slip.assign(liability = p-q/b)
    slip = slip[~slip.liability.isna()]
    slip.liability = slip.liability.clip(lower=0) * bet_size
    slip = slip[slip.liability > 0]

    return create_bets(slip, "pct")

In [25]:
def harville_odds(runners):
    p = runners.normalised_odds
    n=len(p)
    places = runners.place_n.iloc[0]

    if not (places == 2 or places == 3):
        p[:] = 0
        return p
    
    second = p * ((p / (1-p)).sum() - (p / (1-p)))

    place = p + second

    if places == 2:
        return place

    # broadcast over i, j, k
    p = p.to_numpy()
    pi = p[:, None, None]
    pj = p[None, :, None]
    pk = p[None, None, :]

    third = pj * pk / (1 - pj) * pi / (1 - pj - pk)

    # remove invalid cases
    i = np.arange(n)[:, None, None]
    j = np.arange(n)[None, :, None]
    k = np.arange(n)[None, None, :]

    third[(j == k) | (i == j) | (i == k)] = 0

    place += third.sum(axis=(1,2))

    return place

In [26]:
x = df.set_index(["race_id", df.index]).sort_index()

In [27]:
ho = x.groupby(level=0, as_index=False).apply(harville_odds)

In [28]:
x["harville_odds"] = ho

In [29]:
x = x.reset_index(level=0)

In [74]:
def bet_harville_edge(group=None):
    edge = x.harville_odds * x.place_bsp
    selections = x[edge > 1.25]
    
    p = selections.harville_odds
    q = 1-p
    b = selections.place_bsp-1

    cols = ["race_id", "place_win_lose", "place_bsp"]
    if group:
        cols += [group]
    
    slip = selections[cols]
    slip = slip.rename(columns = {"place_win_lose": "win_lose", "place_bsp": "bsp"})
    slip = slip.assign(liability=p-q/b)
    slip = slip[~slip.liability.isna()]
    slip.liability = slip.liability.clip(lower=0)
    slip = slip[slip.liability > 0]
    slip.liability *= 0.1

    return create_bets(slip, "pct")
    

In [112]:
def sharpe_ratio(profit):
    ret = profit-1
    avg_return = ret.mean()
    std_dev = ret.std()

    return avg_return/std_dev

In [113]:
strategies = {
    "bet-fav": (bet_on_favourite, "abs"),
    "hedge-fav": (hedge_on_favourite, "abs"),
    "bet-harville": (bet_harville_edge, "pct")
}

## All

In [117]:
results = dict()
for strat, (f, agg) in strategies.items():
    bets = f()
    p = bets.groupby(bets.race_time.dt.date).profit
    if agg == "abs" :
        p = p.sum().cumsum()
    else:
        p = p.prod().cumprod()
    results[strat] = p

    if strat == "bet-harville":
        sr = sharpe_ratio(bets.groupby(bets.race_time.dt.to_period("D")).profit.prod())*np.sqrt(365)
        print(f"Sharpe Ratio: {sr}")
pd.DataFrame(results).to_csv(f"{fbase}/all-strats.csv")

Sharpe Ratio: 0.9679148734731492


## Groups

In [118]:
for c in cats:
    sharpe_ratios = dict()
    for strat, (f, agg) in strategies.items():
        bets = f(cats[c][0])
        #print(cats[c][0], bets)
        pt = bets.pivot_table(
            index=bets.race_time.dt.date,
            columns=cats[c][0],
            values="profit",
            aggfunc="sum" if agg == "abs" else "prod",
        ).loc[:, cats[c][1]]
        pt = pt.reindex(sorted(pt.columns), axis=1)
        pt.columns = pt.columns.map(str).str.replace(",", ";")
        
        if strat== "bet-harville":
            sr = sharpe_ratio(pt)*np.sqrt(365)
            pd.DataFrame(sr).to_csv(f"{fbase}/{c}-sharpe.csv")

        if agg == "abs":
            pt = pt.cumsum()
        else:
            pt = pt.cumprod()
            
        pt.to_csv(f"{fbase}/{c}-{strat}.csv")

In [79]:
t = bet_harville_edge()
sharpe_ratio(t.groupby(t.race_time.dt.to_period("Y")).profit.prod())

np.float64(0.5552921205339818)

In [82]:
t.profit.std()

np.float64(0.026926592744193276)

In [98]:
t.profit.cumprod()

selection_id
1291169         0.981989
3153288         0.980677
1466872         0.960312
3157074         0.942253
2446553         0.966512
                ...     
101187589    1138.137050
99374565     1185.541826
89407944     1216.184104
5290871      1160.982122
44049799     1190.106036
Name: profit, Length: 16368, dtype: float64